In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'sans-serif'

In [ ]:
RESULTS_DIR = "results"
FIGURES_DIR = "../../figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

ABLATIONS = [
    ("clf",        r"$\lambda_{\text{clf}}$",         "Ablation - (Cellina) Cell-type classifier"),
    ("disc",       r"$\lambda_{\text{disc}}$",        "Ablation - (Cellina) Domain adversary"),
    ("domain_clf", r"$\lambda_{\text{domain\_clf}}$", "Ablation - (Cellina) Domain classifier"),
    ("graph",      r"$\lambda_{\text{graph}}$",        "Ablation - (Cellina-GAT) Contrastive Loss"),
]

dfs = {}
for key, _, _ in ABLATIONS:
    path = os.path.join(RESULTS_DIR, f"ablation_{key}.csv")
    if os.path.exists(path):
        dfs[key] = pd.read_csv(path)
        print(f"{key}: {len(dfs[key])} rows")
    else:
        print(f"{key}: NOT FOUND ({path})")

## Figure 1 — F1 scores (4 panels)

In [ ]:
F1_PALETTE = {
    "F1_celltype":       "#4C4CB6",
    "F1_spatial_domain": "#C83636",
}
F1_LABEL_MAP = {
    "F1_celltype":       "celltype",
    "F1_spatial_domain": "spatial domain",
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)
flat_axes = axes.flatten()
for ax_i, (key, xlabel, title) in enumerate(ABLATIONS):
    ax = flat_axes[ax_i]
    if key not in dfs:
        ax.set_title(title + "\n(no data)", fontsize=13)
        continue

    df = dfs[key][dfs[key]["metric"].isin(["F1_celltype", "F1_spatial_domain"])].copy()
    lambda_order = sorted(df["lambda"].unique())
    lambda_strs = [f"{lv:.0e}" for lv in lambda_order]
    df["lambda_str"] = df["lambda"].map(lambda x: f"{x:.0e}")
    df["label"] = df["metric"].map(F1_LABEL_MAP)

    palette_mapped = {F1_LABEL_MAP[k]: v for k, v in F1_PALETTE.items()}

    # Individual data points
    sns.stripplot(
        data=df,
        x="lambda_str",
        y="score",
        hue="label",
        palette=palette_mapped,
        order=lambda_strs,
        dodge=True,
        jitter=True,
        alpha=0.3,
        size=3,
        zorder=1,
        ax=ax,
        legend=False,
    )

    # Median line + SE error bars
    sns.pointplot(
        data=df,
        x="lambda_str",
        y="score",
        hue="label",
        palette=palette_mapped,
        order=lambda_strs,
        dodge=True,
        estimator=np.median,
        errorbar="se",
        markers="o",
        linestyles="-",
        capsize=0.08,
        scale=1.2,
        linewidth=0.9,
        zorder=2,
        ax=ax,
    )

    if "1e+00" in lambda_strs:
        idx = lambda_strs.index("1e+00")
        ax.axvline(x=idx, color="gray", linestyle="--", linewidth=1.0,
                   label="unit weight", zorder=0)

    ax.set_ylim(0.2, 0.8)
    ax.yaxis.grid(True, alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)
    sns.despine(ax=ax)

    ax.tick_params(axis='x', labelsize=10, rotation=45)
    ax.tick_params(axis='y', labelsize=10)
    ax.set_xlabel(xlabel, fontsize=13)
    ax.set_title(title, fontsize=13)

    if ax_i % 2 == 0:
        ax.set_ylabel("F1 score", fontsize=12)
    else:
        ax.set_ylabel("")

    handles, labels_leg = ax.get_legend_handles_labels()
    if ax_i == 0:
        leg = ax.legend(handles=handles, labels=labels_leg, title="Label",
                        title_fontsize=10, fontsize=10, loc="lower left", framealpha=0.8)
        leg.get_title().set_fontweight("bold")
    else:
        if ax.get_legend():
            ax.get_legend().remove()

plt.tight_layout()

fig.savefig(os.path.join(FIGURES_DIR, "ablations_f1.svg"), bbox_inches="tight")
plt.show()

## Figure 2 — Marginal log-likelihood (4 panels)

In [ ]:
# fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)
# flat_axes = axes.flatten()

# for ax_i, (key, xlabel, title) in enumerate(ABLATIONS):
#     ax = flat_axes[ax_i]
#     if key not in dfs:
#         ax.set_title(title + "\n(no data)", fontsize=13)
#         continue

#     df = dfs[key][dfs[key]["metric"] == "marginal_ll"].copy()
#     lambda_order = sorted(df["lambda"].unique())
#     lambda_strs = [f"{lv:.0e}" for lv in lambda_order]
#     df["lambda_str"] = df["lambda"].map(lambda x: f"{x:.0e}")

#     # Individual data points
#     sns.stripplot(
#         data=df,
#         x="lambda_str",
#         y="score",
#         order=lambda_strs,
#         color="#444444",
#         jitter=True,
#         alpha=0.3,
#         size=3,
#         zorder=1,
#         ax=ax,
#     )

#     # Median line + SE error bars
#     sns.pointplot(
#         data=df,
#         x="lambda_str",
#         y="score",
#         order=lambda_strs,
#         color="#444444",
#         estimator=np.median,
#         errorbar="se",
#         markers="o",
#         linestyles="-",
#         capsize=0.08,
#         scale=1.2,
#         linewidth=0.9,
#         zorder=2,
#         ax=ax,
#     )

#     if "1e+00" in lambda_strs:
#         idx = lambda_strs.index("1e+00")
#         ax.axvline(x=idx, color="gray", linestyle="--", linewidth=1.0,
#                    label="unit weight", zorder=0)
#         ax.legend(fontsize=10, loc="lower left", framealpha=0.8)

#     ax.yaxis.grid(True, alpha=0.3, linestyle="--")
#     ax.set_axisbelow(True)
#     sns.despine(ax=ax)
#     # ax.set_ylim(-470, -435)

#     ax.tick_params(axis='x', labelsize=10, rotation=45)
#     ax.tick_params(axis='y', labelsize=10)
#     ax.set_xlabel(xlabel, fontsize=13)
#     ax.set_title(title, fontsize=13)

#     if ax_i % 2 == 0:
#         ax.set_ylabel("Marginal LL", fontsize=12)
#     else:
#         ax.set_ylabel("")

# plt.tight_layout()

# fig.savefig(os.path.join(FIGURES_DIR, "ablations_mll.svg"), bbox_inches="tight")
# plt.show()

In [ ]:
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

GOOD_FIT_MIN = -468
GOOD_FIT_MAX = -438
BAD_FIT_THRESH = -490   # visual gap between the two windows

fig = plt.figure(figsize=(12, 10))
outer = GridSpec(2, 2, figure=fig, hspace=0.55, wspace=0.35)

def _add_break_marks(ax_top, ax_bot, d=0.015):
    kw = dict(color='k', clip_on=False, linewidth=0.8, transform=ax_top.transAxes)
    ax_top.plot((-d, +d), (-d, +d), **kw)
    ax_top.plot((1-d, 1+d), (-d, +d), **kw)
    kw['transform'] = ax_bot.transAxes
    ax_bot.plot((-d, +d), (1-d, 1+d), **kw)
    ax_bot.plot((1-d, 1+d), (1-d, 1+d), **kw)

def _clip_to_patch(ax):
    for artist in ax.collections + ax.lines:
        artist.set_clip_on(True)
        artist.set_clip_path(ax.patch)

_pp_kw = dict(estimator=np.median, errorbar="se", markers="o", linestyles="-",
              capsize=0.08, scale=1.2, linewidth=0.9, zorder=2)

for ax_i, (key, xlabel, title) in enumerate(ABLATIONS):
    row, col = divmod(ax_i, 2)
    inner = GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[row, col],
                                    height_ratios=[3, 1], hspace=0.08)
    ax_top = fig.add_subplot(inner[0])  # good fits — larger
    ax_bot = fig.add_subplot(inner[1])  # bad fits  — smaller

    if key not in dfs:
        ax_top.set_title(title + "\n(no data)", fontsize=13)
        ax_top.axis('off'); ax_bot.axis('off')
        continue

    df = dfs[key][dfs[key]["metric"] == "marginal_ll"].copy()
    lambda_order = sorted(df["lambda"].unique())
    lambda_strs = [f"{lv:.0e}" for lv in lambda_order]
    df["lambda_str"] = df["lambda"].map(lambda x: f"{x:.0e}")

    df_good = df[df["score"] >= BAD_FIT_THRESH].copy()
    df_bad  = df[df["score"] <  BAD_FIT_THRESH].copy()

    # Scatter all points on both axes — clipping handles what's visible per window
    for ax in (ax_top, ax_bot):
        sns.stripplot(data=df, x="lambda_str", y="score", order=lambda_strs,
                      color="#444444", jitter=True, alpha=0.3, size=3, zorder=1, ax=ax)

    # Median line: good-fit median on ax_top, bad-fit median on ax_bot
    sns.pointplot(data=df_good, x="lambda_str", y="score", order=lambda_strs,
                  color="#444444", **_pp_kw, ax=ax_top)
    if len(df_bad) > 0:
        sns.pointplot(data=df_bad, x="lambda_str", y="score", order=lambda_strs,
                      color="#444444", **_pp_kw, ax=ax_bot)

    # Unit-weight reference on both axes
    if "1e+00" in lambda_strs:
        idx = lambda_strs.index("1e+00")
        for ax in (ax_top, ax_bot):
            ax.axvline(x=idx, color="gray", linestyle="--", linewidth=1.0,
                       label="unit weight", zorder=0)

    # Y-axis windows
    ax_top.set_ylim(GOOD_FIT_MIN, GOOD_FIT_MAX)
    data_min = df["score"].min()
    ax_bot.set_ylim(data_min * 1.05 if data_min < 0 else data_min - abs(data_min) * 0.05,
                    BAD_FIT_THRESH)

    # Clip to axis bounds
    _clip_to_patch(ax_top)
    _clip_to_patch(ax_bot)

    # Break effect
    ax_top.spines['bottom'].set_visible(False)
    ax_bot.spines['top'].set_visible(False)
    ax_top.tick_params(bottom=False, labelbottom=False)
    ax_bot.tick_params(top=False)
    _add_break_marks(ax_top, ax_bot)

    # Grids and despine
    for ax in (ax_top, ax_bot):
        ax.yaxis.grid(True, alpha=0.3, linestyle="--")
        ax.set_axisbelow(True)
        sns.despine(ax=ax, bottom=(ax is ax_top))

    # Labels
    ax_bot.tick_params(axis='x', labelsize=10, rotation=45)
    ax_bot.tick_params(axis='y', labelsize=10)
    ax_top.tick_params(axis='y', labelsize=10)
    ax_bot.set_xlabel(xlabel, fontsize=13)
    ax_top.set_xlabel("")
    ax_top.set_title(title, fontsize=13)

    if col == 0:
        ax_top.set_ylabel("Marginal LL", fontsize=12)
        ax_bot.set_ylabel("")
    else:
        ax_top.set_ylabel(""); ax_bot.set_ylabel("")

    if ax_i == 0 and "1e+00" in lambda_strs:
        ax_top.legend(fontsize=10, loc="lower left", framealpha=0.8)
    for ax in (ax_top, ax_bot):
        if ax_i != 0 and ax.get_legend():
            ax.get_legend().remove()

fig.savefig(os.path.join(FIGURES_DIR, "ablations_mll.svg"), bbox_inches="tight")
print("Saved figure to:", os.path.join(FIGURES_DIR, "ablations_mll.svg"))
plt.show()